In [ ]:
import re
from typing import Any

from typing_extensions import TypedDict

from kagraph import END, START, StateGraph, prompt_llm
from kagraph.llms import load_llm
from kagraph.tracing import trace

In [ ]:
class ReWOOInput(TypedDict):
    task: str


class ReWOOState(TypedDict, total=False):
    task: str
    plan_string: str
    steps: list[tuple[str, str, str, str]]
    results: dict[str, str]
    result: str

In [ ]:
PLAN_REGEX = re.compile(
    r"Plan:\s*(.*?)\s*(#E\d+)\s*=\s*([A-Za-z_][A-Za-z0-9_]*)\s*\[(.*?)\]",
    re.DOTALL,
)

PLANNER_PROMPT = """For the following task, make plans that can solve the problem step by step.
For each plan, indicate which external tool and tool input should be used to retrieve evidence.
You can store evidence into a variable #E that can be called by later tools.

Use exactly this repeating format:
Plan: <short reasoning for this step>
#E1 = <Tool>[<tool input>]

Available tools:
(1) Google[input]: Search a small local fact base. Use this for specific factual lookup.
(2) LLM[input]: Ask a pretrained LLM to reason from prior evidence and common knowledge.

Rules:
- Use only Google or LLM as the tool name.
- Each Plan must be followed by exactly one #E assignment.
- Use #E variables when later steps depend on earlier evidence.
- Keep the plan to 2 or 3 evidence steps.
- Do not solve the task outside the plan.

Task: {task}
"""

SOLVE_PROMPT = """Solve the following task.

We made a step-by-step plan and retrieved evidence for each step. Use the evidence carefully.

{plan}

Task: {task}

Return the final answer directly, with one short supporting sentence.
"""

In [ ]:
def google_search(query: str) -> str:
    """Search a small local fact base for deterministic ReWOO evidence."""

    normalized = _normalize_query(query)
    if "australian open" in normalized and (
        "winner" in normalized or "champion" in normalized
    ):
        return "The 2024 men's Australian Open singles winner was Jannik Sinner."

    if "hometown" in normalized and ("jannik sinner" in normalized or "sinner" in normalized):
        return "Jannik Sinner's exact hometown is Sesto, South Tyrol, Italy."

    if "hometown" in normalized and "australian open" in normalized:
        return (
            "The relevant 2024 men's Australian Open winner is Jannik Sinner, "
            "whose exact hometown is Sesto, South Tyrol, Italy."
        )

    return (
        "No exact local fact matched the query. Relevant known facts: "
        "Jannik Sinner won the 2024 men's Australian Open; his hometown is commonly "
        "listed as Sesto, South Tyrol, Italy."
    )


def _normalize_query(query: str) -> str:
    query = query.lower()
    query = query.replace("men's", "mens")
    query = re.sub(r"[^a-z0-9#]+", " ", query)
    return re.sub(r"\s+", " ", query).strip()


def _parse_plan(plan_text: str) -> list[tuple[str, str, str, str]]:
    steps = [
        (
            plan.strip(),
            step_name.strip(),
            tool.strip(),
            tool_input.strip(),
        )
        for plan, step_name, tool, tool_input in PLAN_REGEX.findall(plan_text)
    ]
    if not steps:
        raise ValueError(f"Planner output did not match ReWOO plan format:\n{plan_text}")
    return steps


def _current_step(state: ReWOOState) -> int | None:
    results = state.get("results") or {}
    steps = state["steps"]
    if len(results) == len(steps):
        return None
    return len(results) + 1


def _substitute_variables(text: str, results: dict[str, str]) -> str:
    for key, value in results.items():
        text = text.replace(key, value)
    return text


def _evidence_plan(state: ReWOOState) -> str:
    results = state.get("results") or {}
    lines = []
    for plan, step_name, tool, tool_input in state["steps"]:
        substituted_input = _substitute_variables(tool_input, results)
        evidence = results.get(step_name, "")
        lines.append(
            f"Plan: {plan}\n"
            f"{step_name} = {tool}[{tool_input}]\n"
            f"Resolved input: {substituted_input}\n"
            f"Evidence: {evidence}"
        )
    return "\n\n".join(lines)

In [ ]:
def build_graph(llm: Any):
    graph = StateGraph(ReWOOState, input_schema=ReWOOInput)

    def get_plan(state: ReWOOState):
        plan_string = str(prompt_llm(llm, PLANNER_PROMPT.format(task=state["task"])))
        return {
            "plan_string": plan_string,
            "steps": _parse_plan(plan_string),
            "results": {},
        }

    def tool_execution(state: ReWOOState):
        step_number = _current_step(state)
        if step_number is None:
            return {}

        _, step_name, tool, tool_input = state["steps"][step_number - 1]
        results = dict(state.get("results") or {})
        substituted_input = _substitute_variables(tool_input, results)

        if tool == "Google":
            output = google_search(substituted_input)
        elif tool == "LLM":
            output = prompt_llm(llm, substituted_input)
        else:
            raise ValueError(f"Unsupported ReWOO tool {tool!r}")

        results[step_name] = str(output)
        return {"results": results}

    def solve(state: ReWOOState):
        answer = prompt_llm(
            llm,
            SOLVE_PROMPT.format(plan=_evidence_plan(state), task=state["task"]),
        )
        return {"result": str(answer)}

    def route_after_tool(state: ReWOOState):
        return "solve" if _current_step(state) is None else "tool"

    graph.add_node("plan", get_plan)
    graph.add_node("tool", tool_execution)
    graph.add_node("solve", solve)

    graph.add_edge(START, "plan")
    graph.add_edge("plan", "tool")
    graph.add_conditional_edges(
        "tool",
        route_after_tool,
        {
            "tool": "tool",
            "solve": "solve",
        },
    )
    graph.add_edge("solve", END)

    return graph.compile(name="rewoo")

In [ ]:
llm = load_llm("qwen/qwen3-next-80b-a3b-instruct")
app = build_graph(llm)

In [ ]:
app.get_graph().draw_png()

In [ ]:
with trace("ReWOO"):
    result = app.invoke(
        {"task": "What is the exact hometown of the 2024 men's Australian Open winner?"},
        chat_name="KaGraph ReWOO example",
        recursion_limit=20,
    )